# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. The dataset comprises mass spectrometry records sourced from human mast cell extracellular vesicles, represented with a detailed Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Loaded dataset: {metadata.name}\nDescription: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

The Croissant schema may define multiple record sets. Each record set contains fields (columns) with specific `@id` values. We'll list all record set `@id`s and the field IDs of one example set.


In [ ]:
# Access the record sets defined in the metadata
record_sets = getattr(metadata, 'recordSet', [])
if hasattr(record_sets, '__dict__') or hasattr(record_sets, '_fields'):  # Single object
    record_sets = [record_sets]

if len(record_sets) == 0:
    raise RuntimeError("No record sets found in the Croissant metadata. Check the schema/URL and schema structure.")

print(f"Found {len(record_sets)} record sets.")
for idx, record_set in enumerate(record_sets):
    rid = getattr(record_set, '@id', f'index_{idx}')
    print(f"Record set {idx+1}: @id = {rid}  | Name: {getattr(record_set, 'name', '')}")
    # Print fields of the record set
    fields = getattr(record_set, 'field', [])
    if hasattr(fields, '__dict__') or hasattr(fields, '_fields'):
        fields = [fields]
    print("  Field @id(s):")
    for f in fields:
        print(f"    - {getattr(f, '@id', '')}: {getattr(f, 'name', '')}")


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll extract all records for the main record set found above.

In [ ]:
# Collect all record set @ids
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]
record_set_ids = [x for x in record_set_ids if x]

# Prepare a DataFrame for each record set
dataframes = {}
for rid in record_set_ids:
    print(f"\nExtracting records from record set: {rid}")
    records = list(dataset.records(record_set=rid))
    df = pd.DataFrame(records)
    dataframes[rid] = df
    print(f"Columns in {rid}: {df.columns.tolist()}")
    print(df.head())

# For illustration, pick the first available record set
example_rid = record_set_ids[0]
print(f"\nMain DataFrame loaded for record set (@id): {example_rid}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Here, we choose a numeric field (`coverage_percent`, as an example) and perform filtering and normalization.

Replace the `numeric_field_id` and `group_field_id` with the correct column `@id` from the Data Overview above. Typical numeric fields might include coverage percent, number of peptides, molecular weight (`MW`), etc.

In [ ]:
# Assign the DataFrame and check columns
df = dataframes[example_rid]
print(f"Columns in the main DataFrame: {df.columns.tolist()}")

# You may need to adjust these @ids/names for fields
# Try to find a numeric field; fallback to the first float/integer column
numeric_field_id = None
for c in df.columns:
    if any(s in str(c).lower() for s in ['coverage', 'percent', 'mw', 'molecular_weight', 'peptide', 'abundance']):
        numeric_field_id = c
        break
if not numeric_field_id:
    for c in df.select_dtypes(include=['int', 'float']).columns:
        numeric_field_id = c
        break

# If still not found, just pick the first column
if not numeric_field_id:
    numeric_field_id = df.columns[0]
print(f"Using column '{numeric_field_id}' as a numeric analysis field.")

# Convert column to numeric, coerce errors
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
threshold = df[numeric_field_id].quantile(0.9)  # Use top 10% as an example
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (90th percentile):");
print(filtered_df.head())

# Normalize selected field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a categorical field such as 'description' or 'accession' if available
possible_group_fields = [c for c in df.columns if c not in [numeric_field_id] and ('descr' in c.lower() or 'accession' in c.lower() or 'type' in c.lower())]
group_field_id = possible_group_fields[0] if possible_group_fields else df.columns[0]
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Visualize the distribution of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# If grouping was successful, visualize grouped barplot
if 'grouped_df' in locals():
    plt.figure(figsize=(10,5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading, exploring, and processing mass spectrometry data from a Croissant-formatted FAIR dataset using the `mlcroissant` library. The approach leveraged rich metadata via precise `@id` references, allowing robust, reproducible workflows for biomedical data analysis. We encourage deeper domain exploration and more advanced visualization using the field `@id`s documented in the Croissant schema.